In [ ]:
# Parameters
city_key = "ywg"
analysis_feed_id = "current"
save_figures = False
figures_dir = "reports/pr2/figures"
dpi = 200

# PR2: Temporal Route Clustering

This notebook clusters routes by their hourly departure profiles to identify service-pattern groups. The output is descriptive and supports interpretation, not a formal policy ranking.

## 1. Setup & Helpers

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

from ptn_analysis.context.reporting import (
    ensure_report_dirs,
    save_placeholder_figure,
    save_report_figure,
)

ensure_report_dirs("pr2")
figure_output_directory = Path(figures_dir)
figure_output_directory.mkdir(parents=True, exist_ok=True)

from ptn_analysis import TransitContext
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 2. Load Route-Hour Profiles

In [ ]:
ctx = TransitContext.from_defaults(feed_id=analysis_feed_id)
frequency_analyzer = ctx.frequency()
hourly_departure_table = frequency_analyzer.build_hourly_departure_table()

print(f"Hourly rows: {len(hourly_departure_table)}")
display(hourly_departure_table.head())

## 3. Elbow Curve

In [ ]:

if hourly_departure_table.empty:
    hourly_profile_table = pd.DataFrame()
    feature_matrix = None
    save_placeholder_figure("clustering_elbow.png", "Hourly departure data is missing.", figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
else:
    hourly_profile_table = hourly_departure_table.pivot_table(
        index="route_short_name",
        columns="hour",
        values="departures",
        fill_value=0,
    )
    row_totals = hourly_profile_table.sum(axis=1)
    hourly_profile_table = hourly_profile_table.div(row_totals.replace(0, 1), axis=0)

    scaler = StandardScaler()
    feature_matrix = scaler.fit_transform(hourly_profile_table)
    candidate_k_values = [2, 3, 4, 5, 6, 7, 8]
    inertia_values = []
    silhouette_values = []
    for cluster_count in candidate_k_values:
        model = KMeans(n_clusters=cluster_count, random_state=42, n_init=10)
        cluster_labels = model.fit_predict(feature_matrix)
        inertia_values.append(model.inertia_)
        silhouette_values.append(silhouette_score(feature_matrix, cluster_labels))

    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(candidate_k_values, inertia_values, marker="o")
    axes[0].set_title("Elbow Curve")
    axes[0].set_xlabel("Cluster count")
    axes[0].set_ylabel("Inertia")
    axes[1].plot(candidate_k_values, silhouette_values, marker="o", color="#1b9e77")
    axes[1].set_title("Silhouette Score")
    axes[1].set_xlabel("Cluster count")
    axes[1].set_ylabel("Score")
    save_report_figure(figure, "clustering_elbow.png", figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(figure)


## 4. Final Route Clusters

In [ ]:
# Fit final k-means on route hourly departure profiles
if not hourly_departure_table.empty:
    final_cluster_count = 4
    final_model = KMeans(n_clusters=final_cluster_count, random_state=42, n_init=10)
    final_labels = final_model.fit_predict(feature_matrix)
    hourly_profile_table = hourly_profile_table.copy()
    hourly_profile_table["cluster"] = final_labels
    cluster_profile_table = hourly_profile_table.groupby("cluster").mean().T
    cluster_route_table = hourly_profile_table[["cluster"]].reset_index()
    display(cluster_route_table.head())
else:
    cluster_profile_table = None
    cluster_route_table = None


## 5. Neighbourhood Service Clustering

Cluster neighbourhoods by transit service features to identify service typologies across Winnipeg.

In [ ]:
import contextily as cx
import geopandas as gpd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

ca = ctx.coverage()
density_table = ca.neighbourhood_density()
jobs_table = ca.jobs_access()

nb_features = density_table.merge(
    jobs_table[["neighbourhood", "jobs_access_score"]],
    on="neighbourhood", how="left"
).fillna(0)

has_temporal = cluster_profile_table is not None
has_spatial = not nb_features.empty and len(nb_features) >= 4

# Shared 4-color palette across both panels
CLUSTER_PALETTE = {0: "#1f77b4", 1: "#ff7f0e", 2: "#2ca02c", 3: "#d62728"}

if not has_temporal and not has_spatial:
    save_placeholder_figure(
        "clustering_combined.png", "Clustering data not available.",
        figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures,
    )
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                             gridspec_kw={"width_ratios": [1, 1.1]})

    # LEFT: Route temporal clusters
    ax_left = axes[0]
    if has_temporal:
        cluster_names = {0: "Flat all-day", 1: "Peak-only", 2: "Extended-evening", 3: "Low-frequency"}
        for cid in cluster_profile_table.columns:
            label = cluster_names.get(cid, f"Cluster {cid}")
            n_routes = (cluster_route_table["cluster"] == cid).sum()
            ax_left.plot(
                cluster_profile_table.index, cluster_profile_table[cid],
                label=f"{label} (n={n_routes})", linewidth=2.5,
                color=CLUSTER_PALETTE.get(cid),
            )
        ax_left.set_xlabel("Hour of day", fontsize=11)
        ax_left.set_ylabel("Share of daily departures", fontsize=11)
        ax_left.legend(fontsize=10, loc="upper right")
        ax_left.grid(alpha=0.3)
    else:
        ax_left.text(0.5, 0.5, "No temporal data", ha="center", va="center")

    # RIGHT: Neighbourhood service clusters map
    ax_right = axes[1]
    if has_spatial:
        feature_cols = ["stop_count", "stop_density_per_km2", "jobs_access_score"]
        X_nb = StandardScaler().fit_transform(nb_features[feature_cols])
        km_nb = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_nb)
        nb_features["service_cluster"] = km_nb.labels_

        neigh_gdf = ctx.working_db.neighbourhood_gdf()
        map_gdf = neigh_gdf.merge(
            nb_features[["neighbourhood", "service_cluster", "stop_density_per_km2"]],
            on="neighbourhood", how="left",
        ).to_crs(epsg=3857)

        cluster_summary = nb_features.groupby("service_cluster")[feature_cols].mean().round(1)
        for c in sorted(nb_features["service_cluster"].unique()):
            row = cluster_summary.loc[c]
            n = (nb_features["service_cluster"] == c).sum()
            label = f"C{c}: {n} nb, {row['stop_density_per_km2']:.0f} stops/km\u00b2"
            subset = map_gdf[map_gdf["service_cluster"] == c]
            subset.plot(ax=ax_right, color=CLUSTER_PALETTE.get(c, "#bdbdbd"),
                        edgecolor="gray", linewidth=0.3, alpha=0.7, label=label)
        map_gdf[map_gdf["service_cluster"].isna()].plot(
            ax=ax_right, color="lightgray", edgecolor="gray", linewidth=0.3, alpha=0.5)

        # Neighbourhood labels: label larger neighbourhoods so reader can orient
        labeled = map_gdf.dropna(subset=["service_cluster"]).copy()
        labeled["centroid_pt"] = labeled.geometry.centroid
        labeled["area_m2"] = labeled.geometry.area
        # Top 5 per cluster by area (ensures geographic spread)
        top_labels = labeled.groupby("service_cluster").apply(
            lambda g: g.nlargest(4, "area_m2"), include_groups=False
        ).reset_index(drop=True)
        # Also add top 3 by stop density that aren't already included
        dense_labels = labeled.nlargest(6, "stop_density_per_km2")
        top_labels = pd.concat([top_labels, dense_labels]).drop_duplicates(subset="neighbourhood")
        for _, row in top_labels.iterrows():
            pt = row["centroid_pt"]
            name = row["neighbourhood"]
            short = name if len(name) <= 16 else name[:14] + ".."
            ax_right.annotate(
                short, (pt.x, pt.y), fontsize=4.5, ha="center", va="center",
                fontweight="bold", color="#222222",
                bbox=dict(boxstyle="round,pad=0.08", facecolor="white", alpha=0.8, edgecolor="none"),
            )

        cx.add_basemap(ax_right, source=cx.providers.CartoDB.Positron, zoom=12)
        ax_right.legend(loc="lower left", fontsize=9)
        ax_right.set_axis_off()
    else:
        ax_right.text(0.5, 0.5, "No spatial data", ha="center", va="center")

    fig.subplots_adjust(top=0.95, bottom=0.02, left=0.06, right=0.99, wspace=0.08)
    save_report_figure(fig, "clustering_combined.png",
                       figures_dir=figure_output_directory, dpi=dpi, enabled=save_figures)
    plt.close(fig)